# 06 - Establish the home reference


Run cells from top to bottom in a fresh Python kernel. This notebook stands alone.
Install and launch as described in [the tutorial index](README.md).
Hardware access is disabled by default. Set your own eight-digit serial; no unit is
selected automatically. Close other notebooks' connections and other control software
before accessing the device. One connection owns the USB receive path.


**Physical motion:** inspect cable slack, attached optics, and the complete possible
rotation path first. Existing direction/backlash settings determine the route; a small
angle difference does not guarantee a short path. Home/move may enable a disabled motor.
The example does not automatically home, retry, or return to zero.

Calls block until completion. Interrupting the cell normally delivers `KeyboardInterrupt`,
which triggers a profiled-stop attempt on the owning connection. Kernel shutdown, force
termination, a lost USB link, or interrupt delivery failure cannot guarantee a stop.
Use your apparatus's physical safety procedure if control is lost. A controller-reported
stop is not independent proof of mechanical rest. Closing USB leaves motor holding state
unchanged.


In [ ]:
from dataclasses import asdict

from k10cr2 import K10CR2

SERIAL = ""  # Copy the serial from discovery and verify it on the mount's label.
CONNECT_TO_DEVICE = False  # Set True only when ready for USB access.


def configured_mount(**settings):
    # Construction validates the serial and timeouts but does not open USB.
    return K10CR2(SERIAL, **settings)

In [ ]:
from k10cr2 import MotionError


def report_motion_error(error):
    print("Original failure:", type(error.original).__name__, str(error.original))
    print("Stop cleanup:", asdict(error.cleanup))
    if not error.cleanup.confirmed_stopped:
        print("PHYSICAL STATE UNKNOWN: follow the apparatus safety procedure.")
    print("Close and explicitly reconnect; inspect state before deciding on another motion.")

## Review state first

This cell only queries. Homing may travel beyond a nearby target; make room for the entire
homing path. A reported existing home reference does not prevent a new explicit home command.


In [ ]:
if CONNECT_TO_DEVICE:
    with configured_mount() as mount:
        print("Identity:", asdict(mount.info()))
        print("Status:", asdict(mount.status()))
        print("Position:", asdict(mount.position()))
else:
    print("Preflight queries skipped.")

## Explicitly home once

Leave the confirmation empty for an offline run. After reviewing the apparatus, enter
`HOME ` followed by your serial (including the space). Re-running this cell with the
confirmation present issues another home. Clear it after use.

Completion requires a new homing/not-homed transition, final homed and idle status, and
zero position counts. `completion_received` separately records a home-complete message.


In [ ]:
CONFIRM_HOME = ""  # Example form: "HOME " + SERIAL; type the actual text to opt in.

if CONNECT_TO_DEVICE and SERIAL and CONFIRM_HOME == f"HOME {SERIAL}":
    with configured_mount() as mount:
        try:
            result = mount.home(timeout=120)
            print(asdict(result))
            print("Fresh position:", asdict(mount.position()))
        except MotionError as error:
            report_motion_error(error)
            raise
else:
    print("Homing skipped; it requires connection opt-in and the exact HOME confirmation.")

On failure, the library attempts profiled-stop cleanup and preserves both outcomes in
`MotionError`. Do not automatically replay the home. Reconnect explicitly and inspect
state before choosing what to do next.

Next: [07 - Absolute motion](07_absolute_motion.ipynb).
